# v4: retrain on ~36k Claude-labelled listening posts (all clients) + Codex labels

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells **in order** (Shift+Enter). Cell 2 asks for `sentiment_colab_v4.zip`.
3. The last cell downloads `student_v4.zip` (~45 MB). Send its path back.

Total time on a T4: about 30–40 minutes. Keep this tab open while it runs.

In [ ]:
# 1. Check the GPU (should print Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload sentiment_colab_v4.zip and unpack it (safe to re-run)
%cd /content
from google.colab import files
uploaded = files.upload()
!rm -rf /content/work && mkdir -p /content/work && unzip -q /content/sentiment_colab_v4.zip -d /content/work
!ls /content/work /content/work/data/silver

In [ ]:
# 3. Install the two packages Colab doesn't have
!pip install -q onnx onnxruntime

In [ ]:
# 4. Train v4 (12 layers, 128 tokens). Full log: /content/train_v4.log  (~25-35 min)
%cd /content/work
!SENTIMENT_DATA_ROOT=/content/work python -u scripts/train_student.py     --layers 12 --max-len 128 --epochs 3 --batch 32 --lr 5e-5     --llm-labels data/silver/llm_codex.jsonl --silver-mode none     --labelled data/silver/claude_listening_labels.jsonl     --out-dir /content/work/student_v4 > /content/train_v4.log 2>&1
!grep -E "labelled posts|real posts|with TweetEval|vocab|training |val accuracy|best val|saved|Error|Traceback" /content/train_v4.log

In [ ]:
# 5. Package the model and download it
import os, zipfile
from google.colab import files
names = ['config.json', 'model.int8.onnx', 'tokenizer.json']
folder = '/content/work/student_v4'
if all(os.path.exists(f'{folder}/{n}') for n in names):
    with zipfile.ZipFile('/content/student_v4.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for n in names:
            z.write(f'{folder}/{n}', f'student_v4/{n}')
    print('size MB:', round(os.path.getsize('/content/student_v4.zip') / 1e6, 1))
    files.download('/content/student_v4.zip')
else:
    print('Model not found. Paste the output of:  !tail -30 /content/train_v4.log')